# Crop Yield + Climate Fine-Tuning (Mistral 7B)

Fine-tunes Mistral-7B-Instruct on a crop-climate dataset using QLoRA via Unsloth. Given country, year, CO2 level, temperature anomaly, and crop list, the model generates a structured agricultural analysis.

**Before running:**
1. Runtime → Change runtime type → GPU → T4
2. Open the Secrets panel (🔑 left sidebar) and add:
   - `HF_TOKEN` — your token from https://huggingface.co/settings/tokens
   - `HF_USERNAME` — your Hugging Face username
3. Upload `train.jsonl` and `val.jsonl` when prompted in cell 4
4. Run cells top to bottom

| Step | Approx. time on free T4 |
|------|--------------------------|
| Install packages | 3–5 min |
| Load model | 3–4 min |
| Dataset prep | < 1 min |
| Training (2 epochs) | 55–75 min |
| Push to Hub | 25–35 min |


In [ ]:
# 1. Install dependencies
# Unsloth pins specific torch/xformers versions so let it handle resolution
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes datasets huggingface_hub -q
print('packages ready')


In [ ]:
# 2. Check GPU
import torch
assert torch.cuda.is_available(), 'No GPU — go to Runtime > Change runtime type > GPU > T4'
print(torch.cuda.get_device_name(0))
print(f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM')


In [ ]:
# 3. Load credentials from Colab Secrets
from google.colab import userdata

HF_TOKEN    = userdata.get('HF_TOKEN')
HF_USERNAME = userdata.get('HF_USERNAME')
REPO_NAME   = 'crop-climate-mistral-7b'

assert HF_TOKEN,    'Add HF_TOKEN to Colab Secrets'
assert HF_USERNAME, 'Add HF_USERNAME to Colab Secrets'
print('credentials loaded')


In [ ]:
# 4. Get dataset files
# Option A: drag train.jsonl and val.jsonl into the Files panel on the left

# Option B: pull from a HuggingFace dataset repo
# from huggingface_hub import login, hf_hub_download
# login(token=HF_TOKEN)
# for fname in ['train.jsonl', 'val.jsonl']:
#     hf_hub_download(
#         repo_id=f'{HF_USERNAME}/crop-climate-data',
#         filename=fname,
#         repo_type='dataset',
#         local_dir='.'
#     )

import os
assert os.path.exists('train.jsonl'), 'train.jsonl not found — upload it first'
assert os.path.exists('val.jsonl'),   'val.jsonl not found — upload it first'
print('dataset files found')


In [ ]:
# 5. Load Mistral 7B in 4-bit (QLoRA) — fits in T4's 15 GB VRAM
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit',
    max_seq_length = 512,
    dtype          = None,
    load_in_4bit   = True,
)
print('model loaded')


In [ ]:
# 6. Attach LoRA adapters
# r=16 gives a good quality/VRAM tradeoff on T4
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                       'gate_proj','up_proj','down_proj'],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
model.print_trainable_parameters()


In [ ]:
# 7. Load and format dataset
from datasets import load_dataset

raw = load_dataset('json', data_files={
    'train':      'train.jsonl',
    'validation': 'val.jsonl',
})

def format_chat(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {'text': texts}

dataset = raw.map(format_chat, batched=True, remove_columns=['messages'])

# drop examples longer than max_seq_length to avoid truncation surprises
MAX_TOKENS = 512
def count_tokens(examples):
    return {'n_tokens': [len(tokenizer.encode(t)) for t in examples['text']]}

dataset = dataset.map(count_tokens, batched=True)

for split in ['train', 'validation']:
    before = len(dataset[split])
    dataset[split] = dataset[split].filter(lambda x: x['n_tokens'] <= MAX_TOKENS)
    after  = len(dataset[split])
    print(f'{split}: {after} kept, {before - after} dropped (>{MAX_TOKENS} tokens)')

print()
print('Sample (first 300 chars):')
print(dataset['train'][0]['text'][:300])


In [ ]:
# 8. Configure trainer
# SFTConfig is required here — TrainingArguments causes a PicklingError
# when Unsloth's compiled trainer tries to save checkpoints
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = './crop-climate-mistral',
    num_train_epochs            = 2,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,       # effective batch size = 8
    warmup_steps                = 50,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 25,
    eval_strategy               = 'steps',
    eval_steps                  = 200,
    save_strategy               = 'steps',
    save_steps                  = 200,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    optim                       = 'adamw_8bit',
    weight_decay                = 0.01,
    lr_scheduler_type           = 'cosine',
    seed                        = 42,
    report_to                   = 'none',
    dataloader_pin_memory       = False,
    dataset_text_field          = 'text',
    max_seq_length              = 512,
    packing                     = False,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset  = dataset['validation'],
    args          = training_args,
)
print('trainer ready')


In [ ]:
# 9. Train
# ~55-75 min on free T4 — keep the browser tab active
trainer_stats = trainer.train()
print('training done')
print(f'  loss:    {trainer_stats.metrics["train_loss"]:.4f}')
print(f'  runtime: {trainer_stats.metrics["train_runtime"]/60:.1f} min')


In [ ]:
# 10. Quick inference test
FastLanguageModel.for_inference(model)

test_messages = [{
    'role': 'user',
    'content': (
        'You are an agricultural analyst. Given the following data for India '
        'in 2015, analyze the crop yields and climate context.\n\n'
        'Country: India\nYear: 2015 (2010s)\n'
        'Global CO2: 401.00 ppm (Critical CO2 era (> 400 ppm))\n'
        'Temperature anomaly: 0.90°C (Warm year)\n'
        'Population: 1,309,000,000\n\n'
        'Crops recorded this year: wheat, rice, maize, potatoes\n\n'
        'Provide a structured agricultural analysis including yield performance, '
        'climate impact, and key observations.'
    )
}]

prompt = tokenizer.apply_chat_template(
    test_messages, tokenize=False, add_generation_prompt=True
)
inputs  = tokenizer([prompt], return_tensors='pt').to('cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    temperature    = 0.7,
    do_sample      = True,
    pad_token_id   = tokenizer.eos_token_id,
)
generated = outputs[0][inputs['input_ids'].shape[-1]:]
print(tokenizer.decode(generated, skip_special_tokens=True))


In [ ]:
# 11. Push merged model to Hugging Face Hub
# This merges the LoRA weights back into the base model and uploads the full 16-bit model
from huggingface_hub import login
login(token=HF_TOKEN)

model.save_pretrained_merged(
    f'{HF_USERNAME}/{REPO_NAME}',
    tokenizer,
    save_method = 'merged_16bit',
)
print(f'model live at: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')


## Done

The fine-tuned model is on Hugging Face. A few things worth doing next:
- Write a model card on HF describing the dataset, training setup, and example outputs
- Link the repo from your GitHub README
- Step 4: wrap this in a FastAPI endpoint
